# Week 3 lab: find what the file is hiding

A file lands on your desk. Someone will make a decision with it at the next
shift handover. Your job today is not to clean it. Your job is to find out what
it will not support, and to leave evidence another person can check.

Three rules hold for the whole lab:

1. **Never edit the raw file.** A fault you delete is a fault nobody else can see.
2. **Report, do not repair.** A correction needs a source, an approver, and a record.
3. **Name your evidence.** "Row 44" beats "some temperatures look high".

Your file is generated from your username, so the faults in it are yours. The
person next to you has different ones. Comparing answers is useful; copying them
will not work.

## Setup

Run this in the terminal first, using your own GitHub Classroom username:

```bash
python src/make_variant.py --student-id your-github-username
```

Then set `STUDENT_ID` below to the same value.

In [1]:
from pathlib import Path
import hashlib
import sys

import pandas as pd
from IPython.display import display

# Put your GitHub Classroom username here. Your file is generated from it.
STUDENT_ID = "MuhammadAsad-77"


def locate_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "src" / "validate_data.py").exists():
            return candidate
    raise FileNotFoundError("Open Jupyter from the week03 project folder.")


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(65536), b""):
            digest.update(block)
    return digest.hexdigest()


PROJECT_ROOT = locate_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from make_variant import slugify  # noqa: E402

DATA_PATH = PROJECT_ROOT / "data" / "raw" / f"shift_log_{slugify(STUDENT_ID)}.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH.name} is missing. Run this in the terminal first:\n"
        f"    python src/make_variant.py --student-id {STUDENT_ID}"
    )

df = pd.read_csv(DATA_PATH)
print(f"File:    {DATA_PATH.name}")
print(f"Rows:    {len(df)}   Columns: {len(df.columns)}")
print(f"SHA-256: {file_sha256(DATA_PATH)}")
print("\nCopy the file name and fingerprint into docs/data_quality_report.md now.")


File:    shift_log_muhammadasad-77.csv
Rows:    72   Columns: 8
SHA-256: 3738f7285c8c295ae0871547855a1dd098de664ba9593502f76eb090b495ab6f

Copy the file name and fingerprint into docs/data_quality_report.md now.


## Step 1. Write the contract before you look

Before any code, answer these in the cell below. You are writing the standard
that the rest of the lab measures the file against. If you skip this, every
check you write later is just an opinion.

- What does one row represent?
- Which columns together identify one row uniquely?
- How many records should each line have, and over what period?
- Which columns can never be negative, and why is that true from their meaning
  rather than from the numbers you happen to see?

**Your contract:**

- One row represents: One production line operating during a single hourly period across three lines (Line_A, Line_B, Line_C) over 24 simulated hours.
- The business key is: `["timestamp", "line_id"]` (timestamp and line identifier together uniquely identify each hourly observation).
- Expected coverage: Exactly 24 distinct hourly observations for each of the three lines (72 distinct records total), covering 2026-01-12 06:00 to 2026-01-13 05:00.
- Columns that cannot be negative, and why: `units_produced` (physical manufacturing unit output cannot be negative), `downtime_min` (equipment stoppage duration within an hour must be between 0 and 60 minutes), `energy_kwh` (active electrical power consumed by running motors is non-negative), and `rejected_units` (defective units count cannot be negative and cannot exceed `units_produced`).


In [2]:
display(df.head(6))

schema = pd.DataFrame({
    "column": df.columns,
    "dtype_on_load": [str(value) for value in df.dtypes],
    "missing": [int(df[column].isna().sum()) for column in df.columns],
    "distinct": [int(df[column].nunique(dropna=True)) for column in df.columns],
})
display(schema)

,timestamp,line_id,shift,units_produced,downtime_min,energy_kwh,rejected_units,motor_temp_c
0,2026-01-12T06:00,Line_A,Day,389,10.7,160.5,5,77.1
1,2026-01-12T06:00,Line_B,Day,359,14.3,146.6,12,75.3
2,2026-01-12T06:00,Line_C,Day,387,3.8,170.6,3,74.2
3,2026-01-12T07:00,Line_A,Day,397,14.9,178.4,7,74.3
4,2026-01-12T07:00,Line_B,Day,391,12.5,161.9,7,75.4
5,2026-01-12T07:00,Line_C,Day,397,11.6,171.5,3,74.3


,column,dtype_on_load,missing,distinct
0,timestamp,object,0,25
1,line_id,object,0,3
2,shift,object,0,2
3,units_produced,int64,0,47
4,downtime_min,float64,0,61
5,energy_kwh,float64,1,65
6,rejected_units,int64,0,14
7,motor_temp_c,float64,0,37


## Step 2. Hunt, before the validator helps you

Work through the four cells below **without running the validator**. You are
looking for records you would not want a supervisor to act on.

Keep a list as you go. You will be asked to commit to it in a moment.

### Completeness

In [3]:
# Check missing values per column
missing_per_col = df.isna().sum()
print("Missing values per column:")
print(missing_per_col[missing_per_col > 0])

missing_rows = df[df.isna().any(axis=1)]
print(f"\nAffected records ({len(missing_rows)}):")
display(missing_rows)


Missing values per column:
energy_kwh    1
dtype: int64

Affected records (1):


,timestamp,line_id,shift,units_produced,downtime_min,energy_kwh,rejected_units,motor_temp_c
54,2026-01-13T00:00,Line_B,Night,379,20.6,NaN,8,73.3


### Uniqueness

In [4]:
# Exact duplicate rows
exact_dupes = df[df.duplicated(keep=False)]
print(f"Exact row duplicates ({len(exact_dupes)} rows):")
display(exact_dupes)

# Business key duplicates (timestamp, line_id)
key_dupes = df[df.duplicated(subset=["timestamp", "line_id"], keep=False)]
print(f"\nBusiness key ['timestamp', 'line_id'] duplicates ({len(key_dupes)} rows):")
display(key_dupes)


Exact row duplicates (2 rows):

Business key ['timestamp', 'line_id'] duplicates (2 rows):


,timestamp,line_id,shift,units_produced,downtime_min,energy_kwh,rejected_units,motor_temp_c
11,2026-01-12T10:00,Line_A,Day,445,7.5,171.3,5,76.1
71,2026-01-12T10:00,Line_A,Day,445,7.5,171.3,5,76.1


,timestamp,line_id,shift,units_produced,downtime_min,energy_kwh,rejected_units,motor_temp_c
11,2026-01-12T10:00,Line_A,Day,445,7.5,171.3,5,76.1
71,2026-01-12T10:00,Line_A,Day,445,7.5,171.3,5,76.1


### Validity

Separate two ideas that feel similar and are not:

- **impossible** under the contract, such as rejecting more units than you made;
- **surprising**, such as a high motor temperature, which may be an error, a
  real event, or a line running differently.

Only the first kind can be judged from the file alone.

In [5]:
display(df.describe())

# Impossible under data contract:
# - downtime_min outside [0, 60]
# - energy_kwh <= 0
# - rejected_units < 0 or rejected_units > units_produced
impossible_mask = (
    (df["downtime_min"] < 0) | (df["downtime_min"] > 60) |
    (df["energy_kwh"] <= 0) |
    (df["rejected_units"] < 0) | (df["rejected_units"] > df["units_produced"])
)
print("Impossible under data contract:")
display(df[impossible_mask])

# Surprising but physically possible (judgement case):
surprising_mask = ~df["motor_temp_c"].between(40.0, 100.0)
print("\nSurprising but physically possible (review threshold):")
display(df[surprising_mask])


Impossible under data contract:

Surprising but physically possible (review threshold):


,units_produced,downtime_min,energy_kwh,rejected_units,motor_temp_c
count,72.000000,72.000000,71.000000,72.000000,72.000000
mean,398.583333,11.193056,162.605634,6.527778,75.433333
std,28.532043,10.970413,10.073613,3.556277,3.819372
min,348.000000,1.000000,139.500000,2.000000,72.000000
25%,379.000000,6.625000,156.350000,4.000000,74.300000
50%,396.500000,9.300000,163.400000,5.500000,75.200000
75%,424.250000,13.775000,169.550000,8.000000,75.825000
max,452.000000,91.300000,182.600000,15.000000,105.400000


,timestamp,line_id,shift,units_produced,downtime_min,energy_kwh,rejected_units,motor_temp_c
22,2026-01-12T13:00,Line_C,Day,387,91.3,166.2,2,75.5


,timestamp,line_id,shift,units_produced,downtime_min,energy_kwh,rejected_units,motor_temp_c
70,13/01/2026 05:00,Line_C,Night,384,8.4,176.7,3,105.4


### Coverage and time

In [6]:
# Timestamp parse check
parsed = pd.to_datetime(df["timestamp"], format="ISO8601", errors="coerce")
unparseable = df[parsed.isna()]
print(f"Unparseable timestamps ({len(unparseable)} rows):")
display(unparseable)

# Coverage per line (24 distinct hours expected)
valid_keys = df.dropna(subset=["timestamp", "line_id"]).drop_duplicates(subset=["timestamp", "line_id"])
print("\nDistinct hourly records per line (expected 24):")
print(valid_keys.groupby("line_id")["timestamp"].count())

# Shift label consistency (Day: 06:00-17:59, Night: 18:00-05:59)
valid_time = parsed.notna()
expected_shift = pd.Series("Night", index=df.index)
expected_shift[(parsed.dt.hour >= 6) & (parsed.dt.hour < 18)] = "Day"
shift_mismatch = df[valid_time & (df["shift"] != expected_shift)]
print(f"\nShift label mismatches ({len(shift_mismatch)} rows):")
display(shift_mismatch)


Unparseable timestamps (1 rows):

Distinct hourly records per line (expected 24):
line_id
Line_A    23
Line_B    24
Line_C    24
Name: timestamp, dtype: int64

Shift label mismatches (0 rows):


,timestamp,line_id,shift,units_produced,downtime_min,energy_kwh,rejected_units,motor_temp_c
70,13/01/2026 05:00,Line_C,Night,384,8.4,176.7,3,105.4


,timestamp,line_id,shift,units_produced,downtime_min,energy_kwh,rejected_units,motor_temp_c


## Step 3. Commit to your list

**Stop here and fill in section 3 of `docs/data_quality_report.md`** with the
records you found, before running the next cell.

This is the point of the exercise. Writing your list down first is what makes
the next cell informative rather than reassuring.

## Step 4. Compare against the checks

The validator runs the rules that are written so far. Unwritten ones are
reported as TODO rather than stopping the run.

In [7]:
from validate_data import validate

payload = validate(DATA_PATH, skip_unwritten=True)
summary = pd.DataFrame(payload["checks"])[["name", "passed", "observed", "affected_rows"]]
display(summary)
print("Not written yet:", payload["unwritten_checks"] or "none")
print("Overall status:", payload["status"])

Not written yet: none
Overall status: FAIL


,name,passed,observed,affected_rows
0,required_columns,True,[],[]
1,timestamp_parse,False,1,[72]
2,required_values_complete,False,1,[56]
3,exact_row_uniqueness,False,2,"[13, 73]"
4,business_key_uniqueness,False,2,"[13, 73]"
5,known_line_ids,True,0,[]
6,positive_energy,True,0,[]
7,rejects_within_output,True,0,[]
8,downtime_range,False,1,[24]
9,motor_temperature_review_range,False,1,[72]


**Answer before moving on:**

- **Which records did the code find that you missed?**
  The missing 08:00 hourly record for Line_A. Looking through existing rows row-by-row cannot easily identify an absent record without explicitly aggregating and checking unique line-hour counts per line.
- **Which did you find that no check covers yet? That is a check you should write:**
  Cross-column operational impossibilities, such as having 60 minutes of full equipment downtime while still producing non-zero units. We added `check_output_during_full_downtime` to prevent this contradiction.
- **Did any single fault trip more than one check? Is that one problem or two?**
  Yes. The non-ISO timestamp format for Line_C at 05:00 (`13/01/2026 05:00` at CSV line 72) failed `timestamp_parse` and affected coverage parsing. Similarly, the duplicated record for Line_A at 10:00 (CSV lines 13 and 73) tripped both `exact_row_uniqueness` and `business_key_uniqueness`. This is a single operational defect (redundant row) tripping multiple checks.


## Step 5. Write the missing checks

Move to the terminal and work in `src/validate_data.py`:

```bash
pytest -q
```

Six checks are marked TODO and six tests are failing. Write one check, re-run,
and repeat until the tests pass. Then come back and run the cell below.

In [8]:
payload = validate(DATA_PATH)
summary = pd.DataFrame(payload["checks"])[["name", "passed", "observed", "affected_rows"]]
display(summary)
print("Failed checks:", payload["failed_checks"])
print("Overall status:", payload["status"])

assert not payload["unwritten_checks"], "Some checks are still TODO. Run pytest -q."
print("\nAll twelve checks are written. Now run the script from the terminal so it")
print("writes artifacts/validation_results.json, and commit that file.")

Failed checks: ['timestamp_parse', 'required_values_complete', 'exact_row_uniqueness', 'business_key_uniqueness', 'downtime_range', 'motor_temperature_review_range', 'expected_line_hour_coverage']
Overall status: FAIL

All twelve checks are written. Now run the script from the terminal so it
writes artifacts/validation_results.json, and commit that file.


,name,passed,observed,affected_rows
0,required_columns,True,[],[]
1,timestamp_parse,False,1,[72]
2,required_values_complete,False,1,[56]
3,exact_row_uniqueness,False,2,"[13, 73]"
4,business_key_uniqueness,False,2,"[13, 73]"
5,known_line_ids,True,0,[]
6,positive_energy,True,0,[]
7,rejects_within_output,True,0,[]
8,downtime_range,False,1,[24]
9,motor_temperature_review_range,False,1,[72]


## Step 6. The row that checks cannot settle

One record in your file is unusual rather than impossible: the high motor
temperature. A check can flag it. A check cannot tell you what it means.

**Answer below:**

- Name three explanations for that reading, at least one of which is not a fault.
- What single piece of evidence would separate them?
- Who holds that evidence?
- Until you have it, what may the handover discussion still use this file for?

**Your answer:**

- **Name three explanations for that reading, at least one of which is not a fault:**
  1. *Real heavy mechanical load or bearing friction surge (not a fault)*: The motor drove an exceptionally dense material batch under maximum permissible torque.
  2. *Sensor drift or thermocouple circuit fault*: Electrical noise, loose wiring, or temperature transducer calibration failure.
  3. *Cooling or ventilation failure*: Auxiliary coolant blower failure or clogged motor housing ventilation vents.
- **What single piece of evidence would separate them?**
  Sub-minute auxiliary SCADA telemetry (motor electrical current draw, vibration FFT spectrum, and drive inverter temperature) combined with physical maintenance inspection notes.
- **Who holds that evidence?**
  The plant shift electrical technician and SCADA systems engineer.
- **Until you have it, what may the handover discussion still use this file for?**
  It may be used for descriptive shift handover review to alert the incoming supervisor to dispatch a physical inspection for Line_C, while refraining from making predictive maintenance or line shutdown decisions.


## Next

1. Finish `docs/data_quality_report.md` with what you now know.
2. Do the attack round with your partner and fill in `docs/attack_log.md`.
3. Make the approval decision in `docs/dataset_approval.md`.
4. Commit and push.